# 13.06 Groq — LPU 고속 추론 카탈로그

`langchain-groq` 패키지는 Groq의 **LPU(Language Processing Unit)** 인프라에서 호스팅되는 오픈 모델(Llama, Mixtral, Whisper 등)을 단일 클래스 `ChatGroq` 로 노출한다. 특화는 **추론 속도**(초당 수백 토큰) — 챗 UX, 음성, 실시간 에이전트에 적합하다.

## 학습 목표

- `langchain-groq` 가 노출하는 chat 클래스 한 개와 지원 모델 family 를 파악한다
- LPU 아키텍처 특유의 처리량(`tok/s`)·과금 단위·tool-calling 호환성을 비교 기준으로 잡는다
- OpenAI/Anthropic 와 동일한 v1 표준 인터페이스(`bind_tools` · structured output) 가 그대로 동작함을 확인한다

## 언제 이 공급자를 고르나

- **체감 지연이 핵심**인 챗 UI · 음성 어시스턴트 (`tok/s` 가 OpenAI 대비 5~10 배)
- 오픈 가중치 모델(Llama 3·3.1, Mixtral, Gemma) 을 호스팅으로 빠르게 실험하고 싶을 때
- Whisper STT 를 같은 키 한 개로 같이 쓰고 싶을 때(별도 OpenAI 키 불필요)
- **참고**: 멀티모달, 긴 컨텍스트(>200k), 최상위 추론 품질이 필요한 워크로드는 Anthropic / OpenAI / Gemini 가 우세

## 13.06.1 환경 설정

필요 키는 `GROQ_API_KEY` 한 개. https://console.groq.com 에서 발급한다. 이 저장소 `.env` 에는 기본 미포함이라 **probe 가 실패하면 catalog 전용 노트북**으로 읽으면 된다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
assert os.environ["GROQ_API_KEY"], "GROQ_API_KEY 가 .env 에 필요합니다"

from langchain_groq import ChatGroq

probe = ChatGroq(model="llama-3.1-8b-instant", max_tokens=64)
print(probe.invoke("ping").content[:80])

## 13.06.2 공급자 제품군 전체 지도

| 영역 | 심볼 / 모델 ID | 비고 / 링크 |
|------|----------------|-------------|
| Chat (Llama family) | `ChatGroq(model="llama-3.3-70b-versatile")` | 균형형, tool-calling OK |
| Chat (Llama instant) | `ChatGroq(model="llama-3.1-8b-instant")` | 최저 지연, 분류·라우팅 |
| Chat (Mixtral) | `ChatGroq(model="mixtral-8x7b-32768")` | 32k context, MoE |
| Chat (Gemma) | `ChatGroq(model="gemma2-9b-it")` | Google 오픈 가중치 |
| Speech-to-text | Whisper Large v3 (REST) | `groq` 공식 SDK 별도, `langchain-groq` 본체에는 미포함 |
| Embeddings | — | Groq 자체 임베딩 미제공. OpenAI/Voyage/Cohere 와 조합 (`02_embeddings/`) |
| Reranker | — | Cohere/Jina/NVIDIA 와 조합 |
| 카테고리 cross-link | `01_chat_models/06_groq.ipynb` | chat 옵션·streaming 깊은 구현 |

Groq 패키지의 본체는 사실상 `ChatGroq` 한 개다. 임베딩·리랭크·벡터 스토어는 다른 공급자와 조합한다.

## 13.06.3 기본 사용 — chat

OpenAI/Anthropic 과 같은 v1 표준 인터페이스. `invoke` · `stream` · `bind_tools` 모두 그대로 동작한다.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=256)
msg = llm.invoke("한국어로 LPU 가 GPU 와 다른 점을 한 문장으로")
print(msg.content)

## 13.06.4 공급자 특화 기능

### LPU 추론 속도

Groq 의 차별점은 **결정론적 dataflow LPU 칩**으로 동일 모델을 GPU 대비 5~10 배 빠르게 디코딩한다. Llama 3.1 8B 기준 **약 750 tok/s**, Llama 3.3 70B 기준 **약 280 tok/s** 가 공개 벤치 수치다(2025 기준).

| 모델 | 처리량 (tok/s, 공개치) | 컨텍스트 | tool calling |
|------|------------------------|----------|--------------|
| `llama-3.1-8b-instant` | ~750 | 128k | ✅ |
| `llama-3.3-70b-versatile` | ~280 | 128k | ✅ |
| `mixtral-8x7b-32768` | ~500 | 32k | ✅ |
| `gemma2-9b-it` | ~500 | 8k | ⚠️ 제한적 |

### tool-calling 호환성

Llama 3.1+ 와 Mixtral 은 OpenAI 스타일 함수 호출을 따른다. `bind_tools(...)` 그대로 사용.

In [ ]:
from langchain_groq import ChatGroq
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).bind_tools([add])
out = llm.invoke("3 더하기 4 는?")
print(out.tool_calls)

### Whisper STT (별도 SDK)

음성→텍스트는 `langchain-groq` 본체가 아니라 공식 `groq` Python SDK 의 `audio.transcriptions.create` 로 호출한다.

```python
from groq import Groq
client = Groq()
with open("audio.m4a", "rb") as f:
    tx = client.audio.transcriptions.create(
        file=("audio.m4a", f.read()),
        model="whisper-large-v3",
        language="ko",
    )
print(tx.text)
```

LangChain 체인에 넣을 때는 `RunnableLambda` 로 래핑하면 자연스럽다.

## 13.06.5 가격·한도·리전

| 항목 | 값 (2025 공개치 기준) |
|------|----------------------|
| 리전 | 단일 리전 (US) — 글로벌 엔드포인트 `api.groq.com` |
| 무료 티어 | 일부 모델 무료, 분당 30 req · 일 1.4M tok 등 모델별 제한 |
| 유료 가격 | Llama 3.1 8B: $0.05 / $0.08 (입력/출력 1M tok), Llama 3.3 70B: $0.59 / $0.79 |
| 컨텍스트 한도 | 모델별 8k~128k (Llama 3.1+ 는 128k) |
| Rate limit | 분당 토큰 한도(TPM) + 분당 요청 한도(RPM). 콘솔에서 모델별 표 확인 |
| SLA | 무료 티어 SLA 없음. 유료 플랜은 99.9% 목표 |

정확한 최신 단가/한도는 https://console.groq.com/docs/rate-limits 와 https://wow.groq.com/pricing/ 에서 확인.

## 핵심 정리

- Groq 공급자는 사실상 `ChatGroq` 한 개로 압축된다 — chat 외 영역(embedding/rerank/vectorstore)은 다른 공급자와 조합
- 차별점은 **속도** — 같은 Llama 모델을 GPU 대비 5~10 배 빠르게 디코딩
- tool-calling · structured output 등 v1 표준 인터페이스는 OpenAI/Anthropic 과 동일
- Whisper STT 는 본체가 아니라 공식 `groq` SDK 로 호출

## 다음

- 채팅 옵션·streaming 깊은 구현: `08_integration/01_chat_models/06_groq.ipynb`
- 다음 공급자 카탈로그: `08_nvidia.ipynb`

## 참고

- 공식 통합 페이지: https://docs.langchain.com/oss/python/integrations/providers/groq
- Groq Console / docs: https://console.groq.com/docs
- LPU 아키텍처 백서: https://wow.groq.com/why-groq/